In [1]:
import numpy as np

states = ["ST1", "S1", "S2", "S3", "ST2"]
actions = ["up", "down"]

terminal_states = {"ST1", "ST2"}

gamma = 0.9
theta = 1e-8

nS = len(states)
nA = len(actions)

def step(s_idx, a_idx):
    s = states[s_idx]

    if s in terminal_states:
        return s_idx, 0.0, True

    if actions[a_idx] == "up":
        next_idx = s_idx - 1
    else:
        next_idx = s_idx + 1

    next_state = states[next_idx]
    reward = 1.0 if next_state == "ST2" else 0.0
    done = next_state in terminal_states

    return next_idx, reward, done


# value iteration for Q*
Q = np.zeros((nS, nA))

while True:
    delta = 0

    Q_new = Q.copy()

    for s in range(nS):
        if states[s] in terminal_states:
            continue

        for a in range(nA):
            next_s, r, done = step(s, a)

            if done:
                target = r
            else:
                target = r + gamma * np.max(Q[next_s])

            Q_new[s, a] = target
            delta = max(delta, abs(Q_new[s, a] - Q[s, a]))

    Q = Q_new

    if delta < theta:
        break


# extract greedy policy
policy = {}

for s in range(nS):
    if states[s] in terminal_states:
        policy[states[s]] = "terminal"
    else:
        best_a = np.argmax(Q[s])
        policy[states[s]] = actions[best_a]


print("Q* values:")
for i, s in enumerate(states):
    print(f"{s}: up={Q[i, 0]:.4f}, down={Q[i, 1]:.4f}")

print("\nGreedy policy:")
for s, a in policy.items():
    print(f"{s}: {a}")

Q* values:
ST1: up=0.0000, down=0.0000
S1: up=0.0000, down=0.8100
S2: up=0.7290, down=0.9000
S3: up=0.8100, down=1.0000
ST2: up=0.0000, down=0.0000

Greedy policy:
ST1: terminal
S1: down
S2: down
S3: down
ST2: terminal
